# 🌡️ Temperature Sensor Fault Detection
## Phase 1: Data Loading & EDA — Temperature Channel Only

---
**Scope:** MPU-9250 onboard temperature sensor only  
**Dataset:** PVS1–PVS9, Left + Right sensors  
**Goal:** Understand the temperature signal before injecting faults in Phase 2  

### Sections
1. Imports & Configuration  
2. Peek at the Data — Find the Temperature Column  
3. Load Temperature from All 9 Datasets  
4. Basic Statistics  
5. Time Series Plots  
6. Distribution Analysis  
7. Left vs Right Comparison  
8. Cross-Dataset Comparison  
9. Save for Phase 2  

---
## 1. Imports & Configuration

Install if needed (run once in terminal):
```
pip install pandas numpy matplotlib seaborn scipy pyarrow psutil
```

In [ ]:
import os, gc, warnings, json
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import gaussian_kde
import psutil

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

# Colors
COLOR_LEFT  = '#E53935'   # red  — temperature feels warm
COLOR_RIGHT = '#1E88E5'   # blue
COLOR_NORM  = '#43A047'   # green — for normal signal reference

def ram():
    vm = psutil.virtual_memory()
    print(f'  RAM: {vm.used/1e9:.1f} / {vm.total/1e9:.1f} GB  ({vm.percent}%)')

print('✅ Imports done.')
ram()

In [ ]:
# ================================================================
#  ONLY CHANGE THIS — your dataset folder path
# ================================================================
BASE_PATH = r'C:\PVS_Dataset'

OUTPUT_DIR  = os.path.join(BASE_PATH, 'outputs_temp_phase1')
CHUNK_SIZE  = 50_000
SAMPLE_N    = 8_000    # rows sampled per file for plots
SAMPLING_HZ = 100

os.makedirs(OUTPUT_DIR, exist_ok=True)

DATASET_META = {
    'PVS1': {'vehicle':'VW Saveiro',  'driver':'Driver 1', 'scenario':'Scenario 1'},
    'PVS2': {'vehicle':'VW Saveiro',  'driver':'Driver 1', 'scenario':'Scenario 2'},
    'PVS3': {'vehicle':'VW Saveiro',  'driver':'Driver 1', 'scenario':'Scenario 3'},
    'PVS4': {'vehicle':'Fiat Bravo',  'driver':'Driver 2', 'scenario':'Scenario 1'},
    'PVS5': {'vehicle':'Fiat Bravo',  'driver':'Driver 2', 'scenario':'Scenario 2'},
    'PVS6': {'vehicle':'Fiat Bravo',  'driver':'Driver 2', 'scenario':'Scenario 3'},
    'PVS7': {'vehicle':'Fiat Palio',  'driver':'Driver 3', 'scenario':'Scenario 1'},
    'PVS8': {'vehicle':'Fiat Palio',  'driver':'Driver 3', 'scenario':'Scenario 2'},
    'PVS9': {'vehicle':'Fiat Palio',  'driver':'Driver 3', 'scenario':'Scenario 3'},
}

if os.path.exists(BASE_PATH):
    print(f'✅ BASE_PATH found: {BASE_PATH}')
else:
    print(f'❌ NOT FOUND: {BASE_PATH}  — update BASE_PATH above')

---
## 2. Peek at the Data — Find the Temperature Column

We read only 5 rows to see the column names. Then identify which column is temperature.

In [ ]:
first_file = os.path.join(BASE_PATH, 'PVS1', 'dataset_mpu_left.csv')
peek = pd.read_csv(first_file, nrows=5)

print('📋 All columns in dataset_mpu_left.csv:')
for i, col in enumerate(peek.columns):
    print(f'  [{i}]  {col}')
print()
display(peek)

In [ ]:
# Auto-detect temperature column
all_cols = list(peek.columns)
TEMP_COL = next((c for c in all_cols if 'temp' in c.lower()), None)

# ================================================================
#  MANUAL OVERRIDE — if auto-detect is wrong, set it here:
# ================================================================
# TEMP_COL = 'Temperature'   # <-- uncomment and edit if needed

if TEMP_COL:
    print(f'✅ Temperature column detected: "{TEMP_COL}"')
    print(f'   Sample values: {peek[TEMP_COL].tolist()}')
else:
    print('❌ Temperature column NOT found automatically.')
    print('   Please set TEMP_COL manually in the override above.')
    print(f'   Available columns: {all_cols}')

---
## 3. Load Temperature from All 9 Datasets

We only read the temperature column — so even though the full CSVs are large,  
we're pulling out just 1 column. This is extremely fast and light on RAM.

In [ ]:
all_samples = []   # EDA samples
all_stats   = []   # Aggregate stats per file
row_counts  = {}

print('🔄 Loading temperature data from all PVS datasets...\n')

for pvs, meta in DATASET_META.items():
    row_counts[pvs] = {}

    for side in ['left', 'right']:
        fp = os.path.join(BASE_PATH, pvs, f'dataset_mpu_{side}.csv')

        if not os.path.exists(fp):
            print(f'  ⚠️  Not found: {fp}')
            continue

        # Read ONLY the temperature column — very memory efficient
        temps = []
        total = 0
        for chunk in pd.read_csv(fp, usecols=[TEMP_COL],
                                  dtype={TEMP_COL: 'float32'},
                                  chunksize=CHUNK_SIZE):
            chunk = chunk.dropna()
            temps.append(chunk[TEMP_COL].values)
            total += len(chunk)
        gc.collect()

        if not temps:
            continue

        full_arr = np.concatenate(temps).astype(np.float32)
        row_counts[pvs][side] = len(full_arr)

        # Aggregate stats
        all_stats.append({
            'dataset_id': pvs, 'side': side,
            'vehicle': meta['vehicle'], 'driver': meta['driver'], 'scenario': meta['scenario'],
            'n'    : len(full_arr),
            'mean' : float(np.mean(full_arr)),
            'std'  : float(np.std(full_arr)),
            'min'  : float(np.min(full_arr)),
            'max'  : float(np.max(full_arr)),
            'rms'  : float(np.sqrt(np.mean(full_arr**2))),
            'range': float(np.max(full_arr) - np.min(full_arr)),
        })

        # Random sample for EDA
        idx = np.random.choice(len(full_arr), size=min(SAMPLE_N, len(full_arr)), replace=False)
        idx.sort()   # keep time order for time series plots
        sample_df = pd.DataFrame({
            TEMP_COL     : full_arr[idx],
            'sample_idx' : idx,
            'dataset_id' : pvs,
            'side'       : side,
            'vehicle'    : meta['vehicle'],
            'driver'     : meta['driver'],
            'scenario'   : meta['scenario'],
        })
        all_samples.append(sample_df)

        print(f'  {pvs} [{side:5s}]  {len(full_arr):>10,} rows  '
              f'mean={np.mean(full_arr):.2f}°C  '
              f'range=[{np.min(full_arr):.1f}, {np.max(full_arr):.1f}]°C')
        del full_arr
        gc.collect()

eda_df = pd.concat(all_samples, ignore_index=True)
stats_df = pd.DataFrame(all_stats)

print(f'\n✅ Done! EDA sample: {len(eda_df):,} rows  |  RAM:')
ram()

---
## 4. Basic Statistics

In [ ]:
print('📊 TEMPERATURE STATISTICS — All Datasets')
print('='*70)

display_df = stats_df[['dataset_id','side','vehicle','scenario',
                         'n','mean','std','min','max','range','rms']].copy()
display_df.columns = ['Dataset','Side','Vehicle','Scenario',
                       'N Samples','Mean (°C)','Std (°C)',
                       'Min (°C)','Max (°C)','Range (°C)','RMS']
for col in ['Mean (°C)','Std (°C)','Min (°C)','Max (°C)','Range (°C)','RMS']:
    display_df[col] = display_df[col].round(4)
display(display_df)

print('\n📌 Summary across all files:')
print(f'  Overall mean temperature : {stats_df["mean"].mean():.2f} °C')
print(f'  Overall std              : {stats_df["std"].mean():.4f} °C')
print(f'  Global min               : {stats_df["min"].min():.2f} °C')
print(f'  Global max               : {stats_df["max"].max():.2f} °C')
print(f'  Max range (any file)     : {stats_df["range"].max():.2f} °C')

print('\n📌 Left vs Right average temperature:')
for side in ['left','right']:
    m = stats_df[stats_df['side']==side]['mean'].mean()
    print(f'  {side:5s} : {m:.4f} °C')

---
## 5. Time Series Plots

What does a **normal** temperature signal look like over time?  
This is our baseline — we need to understand this well before injecting faults.

In [ ]:
# ── Single dataset: Left vs Right overlaid ──
pvs = 'PVS1'
L = eda_df[(eda_df['dataset_id']==pvs) & (eda_df['side']=='left' )][TEMP_COL].values
R = eda_df[(eda_df['dataset_id']==pvs) & (eda_df['side']=='right')][TEMP_COL].values

fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=False)

# Top: overlaid
n = min(len(L), len(R))
axes[0].plot(L[:n], color=COLOR_LEFT,  lw=1.0, alpha=0.85, label='Left sensor')
axes[0].plot(R[:n], color=COLOR_RIGHT, lw=1.0, alpha=0.85, label='Right sensor')
axes[0].set_title(f'Temperature Signal — {pvs} ({DATASET_META[pvs]["vehicle"]}) | Left vs Right',
                   fontsize=12, fontweight='bold')
axes[0].set_ylabel('Temperature (°C)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Bottom: difference between left and right
diff = L[:n] - R[:n]
axes[1].plot(diff, color='#7B1FA2', lw=0.9, alpha=0.8)
axes[1].axhline(0, color='black', lw=1, linestyle='--', alpha=0.5)
axes[1].fill_between(range(len(diff)), diff, 0,
                      where=(diff>0), color=COLOR_LEFT,  alpha=0.2, label='Left warmer')
axes[1].fill_between(range(len(diff)), diff, 0,
                      where=(diff<0), color=COLOR_RIGHT, alpha=0.2, label='Right warmer')
axes[1].set_title('Difference: Left − Right Temperature', fontsize=11)
axes[1].set_ylabel('Δ Temperature (°C)')
axes[1].set_xlabel('Sample Index')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'temp_01_timeseries_lr.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📁 Saved: temp_01_timeseries_lr.png')

In [ ]:
# ── Zoomed view: first 1000 samples ──
# This helps us see fine-grained signal behavior clearly
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(L[:1000], color=COLOR_LEFT,  lw=1.2, alpha=0.9, label='Left')
ax.plot(R[:1000], color=COLOR_RIGHT, lw=1.2, alpha=0.9, label='Right')
ax.set_title(f'Temperature Signal — First 1000 Samples ({pvs}) | Zoomed',
              fontsize=12, fontweight='bold')
ax.set_xlabel('Sample Index (10 samples = 0.1 sec at 100Hz)')
ax.set_ylabel('Temperature (°C)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'temp_02_timeseries_zoom.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📁 Saved: temp_02_timeseries_zoom.png')

In [ ]:
# ── 3x3 grid: temperature signal across all 9 datasets (left side) ──
fig, axes = plt.subplots(3, 3, figsize=(17, 11))
axes = axes.flatten()

for i, (pvs, meta) in enumerate(DATASET_META.items()):
    data = eda_df[(eda_df['dataset_id']==pvs) & (eda_df['side']=='left')][TEMP_COL].values
    axes[i].plot(data, color=COLOR_LEFT, lw=0.8, alpha=0.9)
    mean_val = data.mean()
    axes[i].axhline(mean_val, color='black', lw=1.2, linestyle='--', alpha=0.6,
                     label=f'mean={mean_val:.2f}°C')
    axes[i].set_title(f"{pvs} | {meta['vehicle']}\n{meta['driver']} | {meta['scenario']}",
                       fontsize=9, fontweight='bold')
    axes[i].set_ylabel('Temp (°C)', fontsize=8)
    axes[i].set_xlabel('Sample', fontsize=8)
    axes[i].legend(fontsize=7)
    axes[i].grid(True, alpha=0.3)
    axes[i].tick_params(labelsize=7)

fig.suptitle('Temperature Signal — All 9 Datasets (Left Sensor, EDA Sample)',
              fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'temp_03_all_datasets_grid.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📁 Saved: temp_03_all_datasets_grid.png')

---
## 6. Distribution Analysis

Understanding the **shape** of the temperature distribution is critical —  
it tells us what values are "normal" so we can define fault thresholds in Phase 2.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# 1. Histogram + KDE: Left vs Right (all datasets)
for side, color in [('left', COLOR_LEFT), ('right', COLOR_RIGHT)]:
    d = eda_df[eda_df['side']==side][TEMP_COL].dropna()
    axes[0].hist(d, bins=80, alpha=0.4, color=color, density=True, label=f'{side.capitalize()}')
    kde = gaussian_kde(d.sample(min(len(d), 5000), random_state=42))
    xs  = np.linspace(d.min(), d.max(), 300)
    axes[0].plot(xs, kde(xs), color=color, lw=2.5)
axes[0].set_title('Temperature Distribution\nLeft vs Right (All Datasets)', fontweight='bold')
axes[0].set_xlabel('Temperature (°C)')
axes[0].set_ylabel('Density')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Box plots: one box per dataset (left side)
pvs_list   = list(DATASET_META.keys())
grp_data   = [eda_df[(eda_df['dataset_id']==p) & (eda_df['side']=='left')][TEMP_COL].dropna().values
               for p in pvs_list]
bp = axes[1].boxplot(grp_data, patch_artist=True, labels=pvs_list,
                      boxprops=dict(facecolor=COLOR_LEFT, alpha=0.5),
                      medianprops=dict(color='black', linewidth=2),
                      flierprops=dict(marker='o', markersize=2, alpha=0.3))
axes[1].set_title('Temperature Spread per Dataset\n(Left Sensor)', fontweight='bold')
axes[1].set_xlabel('Dataset')
axes[1].set_ylabel('Temperature (°C)')
axes[1].grid(True, alpha=0.3, axis='y')

# 3. Violin plot: by vehicle
vehicles = eda_df['vehicle'].unique()
vdata    = [eda_df[(eda_df['vehicle']==v) & (eda_df['side']=='left')][TEMP_COL].dropna().values
             for v in vehicles]
vp = axes[2].violinplot(vdata, positions=range(len(vehicles)),
                         showmedians=True, showmeans=True)
for pc in vp['bodies']:
    pc.set_facecolor(COLOR_LEFT)
    pc.set_alpha(0.5)
axes[2].set_xticks(range(len(vehicles)))
axes[2].set_xticklabels([v.replace(' ','-') for v in vehicles], fontsize=9)
axes[2].set_title('Temperature Distribution by Vehicle\n(Left Sensor)', fontweight='bold')
axes[2].set_ylabel('Temperature (°C)')
axes[2].grid(True, alpha=0.3, axis='y')

plt.suptitle('Temperature Distribution Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'temp_04_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📁 Saved: temp_04_distributions.png')

In [ ]:
# ── Normality check — important for fault threshold setting ──
print('📊 NORMALITY & SHAPE ANALYSIS')
print('='*50)

for side in ['left', 'right']:
    d = eda_df[eda_df['side']==side][TEMP_COL].dropna()
    skew = d.skew()
    kurt = d.kurtosis()
    # Shapiro-Wilk on small sample (max 5000 for speed)
    sample = d.sample(min(2000, len(d)), random_state=42)
    stat, p = stats.shapiro(sample)
    print(f'\n  {side.upper()} sensor:')
    print(f'    Skewness  : {skew:.4f}  {"(roughly symmetric)" if abs(skew)<0.5 else "(skewed)"}')
    print(f'    Kurtosis  : {kurt:.4f}  {"(heavy tails)" if kurt>1 else "(normal-like tails)"}')
    print(f'    Shapiro-W : stat={stat:.4f}  p={p:.4f}  {"→ NOT normal" if p<0.05 else "→ Normal-like"}')

print('\n📌 Why this matters for Phase 2:')
print('   Fault injection thresholds will be set based on mean ± k*std.')
print('   If data is non-normal, we use percentile-based thresholds instead.')

---
## 7. Left vs Right Comparison

Do the left and right temperature sensors read the same values?  
If they differ significantly, that itself could be a fault indicator.

In [ ]:
print('📊 LEFT vs RIGHT — Statistical Comparison')
print('='*55)

rows = []
for pvs in DATASET_META:
    L = eda_df[(eda_df['dataset_id']==pvs) & (eda_df['side']=='left' )][TEMP_COL].dropna()
    R = eda_df[(eda_df['dataset_id']==pvs) & (eda_df['side']=='right')][TEMP_COL].dropna()
    t, p = stats.ttest_ind(L, R, equal_var=False)
    rows.append({
        'Dataset'   : pvs,
        'Vehicle'   : DATASET_META[pvs]['vehicle'],
        'L mean (°C)': round(L.mean(), 4),
        'R mean (°C)': round(R.mean(), 4),
        'Diff (°C)' : round(abs(L.mean()-R.mean()), 4),
        'L std'     : round(L.std(),  5),
        'R std'     : round(R.std(),  5),
        'p-value'   : round(p, 5),
        'Significant': 'YES ⚠️' if p < 0.05 else 'NO ✅',
    })

lr_df = pd.DataFrame(rows).set_index('Dataset')
display(lr_df)

print('\n📌 Interpretation:')
print('   If L mean ≈ R mean → sensors are consistent.')
print('   Large systematic difference → suggests mounting position or thermal effect.')
print('   In fault detection: a sudden L vs R divergence = strong fault indicator.')

In [ ]:
# ── Scatter plot: Left temp vs Right temp ──
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, pvs in zip(axes, ['PVS1', 'PVS4', 'PVS7']):
    L = eda_df[(eda_df['dataset_id']==pvs) & (eda_df['side']=='left' )][TEMP_COL].values
    R = eda_df[(eda_df['dataset_id']==pvs) & (eda_df['side']=='right')][TEMP_COL].values
    n = min(len(L), len(R))

    ax.scatter(L[:n], R[:n], alpha=0.15, s=6, color='#7B1FA2')

    # Perfect agreement line
    lo = min(L[:n].min(), R[:n].min())
    hi = max(L[:n].max(), R[:n].max())
    ax.plot([lo, hi], [lo, hi], 'r--', lw=1.8, label='y = x (perfect match)')

    # Correlation
    r, _ = stats.pearsonr(L[:n], R[:n])
    ax.set_title(f'{pvs}: {DATASET_META[pvs]["vehicle"]}\nr = {r:.4f}',
                  fontsize=11, fontweight='bold')
    ax.set_xlabel('Left Temperature (°C)')
    ax.set_ylabel('Right Temperature (°C)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Left vs Right Temperature Agreement', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'temp_05_lr_scatter.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📁 Saved: temp_05_lr_scatter.png')

---
## 8. Cross-Dataset Comparison

How does temperature vary across vehicles, drivers, and scenarios?  
This tells us whether we need separate fault thresholds per dataset, or one global threshold.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Mean temp per dataset (left + right)
left_means  = [stats_df[(stats_df['dataset_id']==p)&(stats_df['side']=='left' )]['mean'].values[0]
                for p in DATASET_META if len(stats_df[(stats_df['dataset_id']==p)&(stats_df['side']=='left')])>0]
right_means = [stats_df[(stats_df['dataset_id']==p)&(stats_df['side']=='right')]['mean'].values[0]
                for p in DATASET_META if len(stats_df[(stats_df['dataset_id']==p)&(stats_df['side']=='right')])>0]
left_stds   = [stats_df[(stats_df['dataset_id']==p)&(stats_df['side']=='left' )]['std'].values[0]
                for p in DATASET_META if len(stats_df[(stats_df['dataset_id']==p)&(stats_df['side']=='left')])>0]
right_stds  = [stats_df[(stats_df['dataset_id']==p)&(stats_df['side']=='right')]['std'].values[0]
                for p in DATASET_META if len(stats_df[(stats_df['dataset_id']==p)&(stats_df['side']=='right')])>0]

x = np.arange(9)
axes[0].errorbar(x-0.15, left_means,  yerr=left_stds,  fmt='o-', color=COLOR_LEFT,
                  capsize=4, lw=2, markersize=7, label='Left',  alpha=0.9)
axes[0].errorbar(x+0.15, right_means, yerr=right_stds, fmt='s-', color=COLOR_RIGHT,
                  capsize=4, lw=2, markersize=7, label='Right', alpha=0.9)
axes[0].set_xticks(x)
axes[0].set_xticklabels(list(DATASET_META.keys()))
axes[0].set_title('Mean Temperature per Dataset\n(error bars = ±1 std)', fontweight='bold')
axes[0].set_ylabel('Temperature (°C)')
axes[0].set_xlabel('Dataset')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Heatmap: mean temp by vehicle x scenario
vehicles  = ['VW Saveiro','Fiat Bravo','Fiat Palio']
scenarios = ['Scenario 1','Scenario 2','Scenario 3']
hmap = np.zeros((3,3))
for i, veh in enumerate(vehicles):
    for j, sc in enumerate(scenarios):
        rows = stats_df[(stats_df['vehicle']==veh) &
                         (stats_df['scenario']==sc)  &
                         (stats_df['side']=='left')]['mean']
        hmap[i,j] = rows.mean() if len(rows)>0 else np.nan

sns.heatmap(hmap, annot=True, fmt='.2f', cmap='YlOrRd',
             xticklabels=['Sc 1','Sc 2','Sc 3'],
             yticklabels=['VW Saveiro','Fiat Bravo','Fiat Palio'],
             linewidths=0.5, ax=axes[1],
             cbar_kws={'label':'Mean Temp (°C)'},
             annot_kws={'size':11})
axes[1].set_title('Mean Temperature Heatmap\nVehicle × Scenario (Left Sensor)', fontweight='bold')

plt.suptitle('Cross-Dataset Temperature Variability', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'temp_06_cross_dataset.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📁 Saved: temp_06_cross_dataset.png')

In [ ]:
# ── Compute fault injection parameters for Phase 2 ──
# These values will be used directly in Phase 2 to calibrate realistic faults

print('📌 FAULT INJECTION PARAMETERS FOR PHASE 2')
print('='*55)
print('(Computed from actual signal statistics)\n')

global_mean = stats_df['mean'].mean()
global_std  = stats_df['std'].mean()
global_min  = stats_df['min'].min()
global_max  = stats_df['max'].max()

print(f'  Global mean : {global_mean:.4f} °C')
print(f'  Global std  : {global_std:.6f} °C')
print(f'  Global range: [{global_min:.2f}, {global_max:.2f}] °C')

print(f'\n  Recommended fault parameters:')
spike_amp = round(global_std * 8, 4)
bias_amt  = round(global_std * 3, 4)
print(f'  Noise/Spike amplitude   : ±{spike_amp:.4f} °C  (= 8 × std)')
print(f'  Bias/Drift offset       : +{bias_amt:.4f} °C  (= 3 × std per 100 samples)')
print(f'  Flat-line value         :  {global_mean:.4f} °C  (= global mean, sensor stuck)')
print(f'  Normal operating range  :  {global_mean - 3*global_std:.4f} to {global_mean + 3*global_std:.4f} °C')

# Save fault parameters
fault_params = {
    'TEMP_COL'          : TEMP_COL,
    'global_mean'       : round(global_mean, 6),
    'global_std'        : round(global_std,  6),
    'global_min'        : round(global_min,  4),
    'global_max'        : round(global_max,  4),
    'spike_amplitude'   : spike_amp,
    'bias_offset'       : bias_amt,
    'flatline_value'    : round(global_mean, 4),
    'normal_lower_bound': round(global_mean - 3*global_std, 4),
    'normal_upper_bound': round(global_mean + 3*global_std, 4),
    'SAMPLING_HZ'       : SAMPLING_HZ,
    'BASE_PATH'         : BASE_PATH,
    'OUTPUT_DIR'        : OUTPUT_DIR,
}

---
## 9. Save for Phase 2

In [ ]:
# EDA sample
eda_path = os.path.join(OUTPUT_DIR, 'temp_eda_sample.parquet')
eda_df.to_parquet(eda_path, index=False)
print(f'✅ EDA sample saved      → {eda_path}')
print(f'   Size: {os.path.getsize(eda_path)/1e6:.2f} MB  |  Rows: {len(eda_df):,}')

# Stats table
stats_path = os.path.join(OUTPUT_DIR, 'temp_statistics.csv')
stats_df.to_csv(stats_path, index=False)
print(f'✅ Statistics saved       → {stats_path}')

# Fault params config for Phase 2
cfg_path = os.path.join(OUTPUT_DIR, 'temp_config.json')
with open(cfg_path, 'w') as f:
    json.dump(fault_params, f, indent=2)
print(f'✅ Fault parameters saved → {cfg_path}')

# Summary
print('\n' + '='*60)
print('PHASE 1 COMPLETE — TEMPERATURE SENSOR EDA')
print('='*60)
total = sum(v for d in row_counts.values() for v in d.values())
print(f'  Total temperature readings loaded : {total:,}')
print(f'  EDA sample used for plots         : {len(eda_df):,}')
print(f'  Temperature column                : {TEMP_COL}')
print(f'  Normal operating range            : {fault_params["normal_lower_bound"]} – {fault_params["normal_upper_bound"]} °C')
print(f'\n  Plots saved:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.endswith('.png'):
        print(f'    {f}')
print(f'\n  ➡️  Open Phase2_Temperature_Fault_Injection.ipynb to continue.')
ram()